In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


RUN_ID = "20260806_181941_27813"  # set to the benchmark run
CSV_DIR = Path("..") / "results" / RUN_ID / "csv"
PLOT_DIR = Path("plots") / RUN_ID
PLOT_DIR.mkdir(parents=True, exist_ok=True)

sweep = pd.read_csv(CSV_DIR / "benchmark_sweep.csv")

"""
Plot latency vs batch size for fp32 vs int8, with the linear overhead/compute
decomposition overlaid. Reveals WHY small models show little speedup (fixed
per-call overhead dominates) while large models show real int8 gains (compute slope).

Reads benchmark_sweep.csv:
  model, dataset, stage, batch, fp32_latency_ms, int8_latency_ms,
  fp32_throughput_ips, int8_throughput_ips, speedup_x
"""

'\nPlot latency vs batch size for fp32 vs int8, with the linear overhead/compute\ndecomposition overlaid. Reveals WHY small models show little speedup (fixed\nper-call overhead dominates) while large models show real int8 gains (compute slope).\n\nReads benchmark_sweep.csv:\n  model, dataset, stage, batch, fp32_latency_ms, int8_latency_ms,\n  fp32_throughput_ips, int8_throughput_ips, speedup_x\n'

In [2]:
def fit_line(x, y):
    # latency ~ intercept + slope * batch; returns slope, intercept, r2
    slope, intercept = np.polyfit(x, y, 1)
    pred = slope * np.asarray(x) + intercept
    ss_res = np.sum((np.asarray(y) - pred) ** 2)
    ss_tot = np.sum((np.asarray(y) - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return slope, intercept, r2


def plot_one(model, dataset, stage):
    sub = sweep[
        (sweep["model"] == model)
        & (sweep["dataset"] == dataset)
        & (sweep["stage"] == stage)
    ].sort_values("batch")
    if sub.empty:
        return None

    b = sub["batch"].to_numpy()
    fp32 = sub["fp32_latency_ms"].to_numpy()
    int8 = sub["int8_latency_ms"].to_numpy()

    fp32_slope, fp32_int, fp32_r2 = fit_line(b, fp32)
    int8_slope, int8_int, int8_r2 = fit_line(b, int8)

    fig, ax = plt.subplots(figsize=(9, 6))

    # raw measured points
    ax.scatter(b, fp32, color="tab:blue", label="fp32 (measured)", zorder=3)
    ax.scatter(b, int8, color="tab:orange", label="int8 (measured)", zorder=3)

    # fitted lines: intercept = fixed overhead, slope = per-sample compute
    bx = np.linspace(b.min(), b.max(), 100)
    ax.plot(bx, fp32_slope * bx + fp32_int, color="tab:blue", alpha=0.6,
            label=f"fp32 fit (overhead={fp32_int:.2f}ms, R²={fp32_r2:.2f})")
    ax.plot(bx, int8_slope * bx + int8_int, color="tab:orange", alpha=0.6,
            label=f"int8 fit (overhead={int8_int:.2f}ms, R²={int8_r2:.2f})")

    ax.set_xlabel("batch size")
    ax.set_ylabel("latency (ms)")
    ax.set_title(f"{model} / {dataset} / {stage}\n"
                 f"compute slope: fp32={fp32_slope:.4f}  int8={int8_slope:.4f} ms/sample")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    out = PLOT_DIR / f"latency_{model}_{dataset}_{stage}.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    return out




In [3]:
# one plot per model/dataset/stage present in the sweep
combos = sweep[["model", "dataset", "stage"]].drop_duplicates()
for _, row in combos.iterrows():
    out = plot_one(row["model"], row["dataset"], row["stage"])
    if out:
        print("saved", out)

saved plots/20260806_181941_27813/latency_cnn_CIFAR10_PTQ.png
saved plots/20260806_181941_27813/latency_cnn_CIFAR10_QAT.png
saved plots/20260806_181941_27813/latency_resnet18_no_weights_CIFAR10_PTQ.png
saved plots/20260806_181941_27813/latency_resnet18_no_weights_CIFAR10_QAT.png
saved plots/20260806_181941_27813/latency_resnet50_no_weights_CIFAR10_PTQ.png
saved plots/20260806_181941_27813/latency_resnet50_no_weights_CIFAR10_QAT.png
saved plots/20260806_181941_27813/latency_cnn_IMAGENET100_PTQ.png
saved plots/20260806_181941_27813/latency_cnn_IMAGENET100_QAT.png
saved plots/20260806_181941_27813/latency_resnet18_no_weights_IMAGENET100_PTQ.png
saved plots/20260806_181941_27813/latency_resnet18_no_weights_IMAGENET100_QAT.png
saved plots/20260806_181941_27813/latency_resnet50_no_weights_IMAGENET100_PTQ.png
saved plots/20260806_181941_27813/latency_resnet50_no_weights_IMAGENET100_QAT.png
